In [0]:
# Criando a função para filtrar os dados do bronze e colocar no silver
# df_silver = df.filter(F.col("rating").rlike(r"^[0-9]\.[0-9]$"))

from pyspark.sql import functions as F

df_bronze = spark.sql("SELECT * FROM files.amazon_files.amazon_bronze")

df_silver = df_bronze.filter(F.col("rating").rlike(r"^[0-9]\.[0-9]$"))

df_silver.count()

1353

In [0]:
# cast = Serve para converter uma coluna para outro tipo de dado, nesse caso para "double"
# double = é um tipo de dado numérico decimal

df_silver = df_silver.withColumn(
    "discounted_price",
    F.regexp_replace(F.col("discounted_price"), "[^0-9.]", "").cast("double")
)

In [0]:
df_silver = df_silver.withColumn(
    "actual_price",
    F.regexp_replace(F.col("actual_price"), "[^0-9.]", "").cast("double")
)

In [0]:
df_silver = df_silver.withColumn(
    "discount_percentage",
    (F.regexp_replace(F.col("discount_percentage"), "[^0-9]", "").cast("double")) / 100
)

In [0]:
df_silver = df_silver.withColumn(
    "rating",
    F.col("rating").cast("double")
)

In [0]:
df_silver = df_silver.withColumn(
    "rating_count",
    F.regexp_replace(F.col("rating_count"), "[^0-9]", "").cast("double")
)

In [0]:
df_silver.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- discounted_price: double (nullable = true)
 |-- actual_price: double (nullable = true)
 |-- discount_percentage: double (nullable = true)
 |-- rating: double (nullable = true)
 |-- rating_count: double (nullable = true)
 |-- about_product: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- user_name: string (nullable = true)
 |-- review_id: string (nullable = true)
 |-- review_title: string (nullable = true)
 |-- review_content: string (nullable = true)
 |-- img_link: string (nullable = true)
 |-- product_link: string (nullable = true)



In [0]:
df_silver.select("discounted_price", "actual_price", "discount_percentage", "rating", "rating_count").show(10)

+----------------+------------+-------------------+------+------------+
|discounted_price|actual_price|discount_percentage|rating|rating_count|
+----------------+------------+-------------------+------+------------+
|           399.0|      1099.0|               0.64|   4.2|     24269.0|
|           199.0|       349.0|               0.43|   4.0|     43994.0|
|           199.0|      1899.0|                0.9|   3.9|      7928.0|
|           329.0|       699.0|               0.53|   4.2|     94363.0|
|           154.0|       399.0|               0.61|   4.2|     16905.0|
|           149.0|      1000.0|               0.85|   3.9|     24871.0|
|          176.63|       499.0|               0.65|   4.1|     15188.0|
|           229.0|       299.0|               0.23|   4.3|     30411.0|
|           499.0|       999.0|                0.5|   4.2|    179691.0|
|           199.0|       299.0|               0.33|   4.0|     43994.0|
+----------------+------------+-------------------+------+------

In [0]:
df_silver.filter(F.col("discounted_price") > F.col("actual_price")).count()

0

In [0]:
df_silver.filter(F.col("discounted_price") > F.col("actual_price")) \
    .select("product_id", "category", "discounted_price", "actual_price", "discount_percentage") \
        .show(13, truncate=False)

+----------+--------+----------------+------------+-------------------+
|product_id|category|discounted_price|actual_price|discount_percentage|
+----------+--------+----------------+------------+-------------------+
+----------+--------+----------------+------------+-------------------+



In [0]:
df_silver.select("discounted_price", "actual_price", "discount_percentage", "rating", "rating_count").show(10)

+----------------+------------+-------------------+------+------------+
|discounted_price|actual_price|discount_percentage|rating|rating_count|
+----------------+------------+-------------------+------+------------+
|           399.0|      1099.0|               0.64|   4.2|     24269.0|
|           199.0|       349.0|               0.43|   4.0|     43994.0|
|           199.0|      1899.0|                0.9|   3.9|      7928.0|
|           329.0|       699.0|               0.53|   4.2|     94363.0|
|           154.0|       399.0|               0.61|   4.2|     16905.0|
|           149.0|      1000.0|               0.85|   3.9|     24871.0|
|          176.63|       499.0|               0.65|   4.1|     15188.0|
|           229.0|       299.0|               0.23|   4.3|     30411.0|
|           499.0|       999.0|                0.5|   4.2|    179691.0|
|           199.0|       299.0|               0.33|   4.0|     43994.0|
+----------------+------------+-------------------+------+------

In [0]:
df_silver.write.format("delta").mode("overwrite").saveAsTable("files.amazon_files.amazon_silver")

In [0]:
spark.sql("SELECT * FROM files.amazon_files.amazon_silver").show(5)

+----------+--------------------+--------------------+----------------+------------+-------------------+------+------------+---------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|product_id|        product_name|            category|discounted_price|actual_price|discount_percentage|rating|rating_count|        about_product|             user_id|           user_name|           review_id|        review_title|      review_content|            img_link|        product_link|
+----------+--------------------+--------------------+----------------+------------+-------------------+------+------------+---------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|B07JW9H4J1|Wayona Nylon Brai...|Computers&Accesso...|           399.0|      1099.0|               0.64|   4.2|     24

In [0]:
spark.sql("SELECT * FROM files.amazon_files.amazon_silver").count()

1353